<a href="https://colab.research.google.com/github/venkat932023/Government-Information-Chatbot/blob/main/Government_Information_Assistant_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# 🇮🇳 Our Government AI Chatbot
# PM India Government Information Assistant
# Google Colab Ready
# ============================================

# Install dependencies
!pip install -q transformers accelerate gradio requests beautifulsoup4 torch

# ============================================
# Imports
# ============================================

import torch
import requests
from bs4 import BeautifulSoup
import gradio as gr

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

# ============================================
# Load Qwen Model
# ============================================

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# ============================================
# Scrape PM India Government Data
# ============================================

def get_government_data():

    url = "https://www.pmindia.gov.in/en/our-government/"

    try:
        response = requests.get(url, timeout=10)

        soup = BeautifulSoup(response.text, "html.parser")

        text = soup.get_text(separator=" ", strip=True)

        # Reduce token size
        text = text[:8000]

        return text

    except Exception as e:
        return f"Error fetching data: {e}"

government_context = get_government_data()

# ============================================
# Professional System Prompt
# ============================================

SYSTEM_PROMPT = f"""
You are "Our Government AI Assistant".

You are a professional AI chatbot for the Government of India.

Your responsibilities:
- Explain Indian government ministries
- Provide information about Prime Minister and cabinet
- Explain government departments
- Guide users professionally
- Answer only government-related questions
- Be respectful and informative

Use the following official government information:

{government_context}

If information is unavailable,
politely say you do not know.
"""

# ============================================
# Chat Function
# ============================================

def generate_response(message, history):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]

    # Add conversation history
    for user_msg, bot_msg in history:
        messages.append({
            "role": "user",
            "content": user_msg
        })

        messages.append({
            "role": "assistant",
            "content": bot_msg
        })

    # Current message
    messages.append({
        "role": "user",
        "content": message
    })

    # Apply Qwen chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )

    model_inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    # Generate output
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    # Remove thinking tokens
    try:
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    content = tokenizer.decode(
        output_ids[index:],
        skip_special_tokens=True
    ).strip()

    return content


# ============================================
# Professional CSS
# ============================================

custom_css = """

body {
    background: #f4f7fb;
}

.gradio-container {
    font-family: Arial, sans-serif;
}

.main-title {
    text-align: center;
    font-size: 38px;
    font-weight: bold;
    color: #0b3d91;
    margin-bottom: 10px;
}

.subtitle {
    text-align: center;
    font-size: 18px;
    color: #555;
    margin-bottom: 25px;
}

.footer {
    text-align: center;
    color: gray;
    font-size: 13px;
    margin-top: 10px;
}
"""

# ============================================
# Example Questions
# ============================================

examples = [
    "Who is the Prime Minister of India?",
    "Tell me about the Ministry of Education",
    "Explain the role of the Home Ministry",
    "What is Digital India?",
    "Who are cabinet ministers?",
    "Explain PMO",
    "Tell me about Indian government departments",
    "What does the Finance Ministry do?"
]

# ============================================
# Build Gradio UI
# ============================================

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=custom_css
) as demo:

    gr.HTML("""
    <div class="main-title">
        🇮🇳 Our Government AI Assistant
    </div>

    <div class="subtitle">
        Official Government Information Assistant <br>
        Powered by Qwen AI + Gradio
    </div>
    """)

    gr.ChatInterface(
        fn=generate_response,

        chatbot=gr.Chatbot(
            height=550,
            show_copy_button=True
        ),

        textbox=gr.Textbox(
            placeholder="Ask about Indian Government...",
            container=True,
            scale=7
        ),

        title="Government Information Chatbot",

        description="""
        Ask questions about:
        • Prime Minister
        • Ministries
        • Cabinet
        • Government Departments
        • Digital India
        • Government Initiatives
        """,

        examples=examples
    )

    gr.HTML("""
    <div class="footer">
        Government Information Assistant • Built using Qwen 0.6B
    </div>
    """)

# ============================================
# Launch App
# ============================================

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/tmp/ipykernel_903/2335641754.py:213: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_903/2335641754.py:213: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_903/2335641754.py:232: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot=gr.Chatbot(
/tmp/ipykernel_903/2335641754.py:232: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot=gr.Chatbot(
/tmp/ipykernel_903/2

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0f42ba8561cd5ca689.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
